In [ ]:
!pip install gptqmodel

In [1]:
%pip install "protobuf<6.30"

In [ ]:
from gptqmodel import GPTQModel

https://huggingface.co/collections/ModelCloud/vortex

In [ ]:
model = GPTQModel.load("ModelCloud/Llama-3.2-1B-Instruct-gptqmodel-4bit-vortex-v2.5") # Pre Quantized model

In [4]:
result = model.generate("What is the capital of France?")
print(result)

tensor([[128000,   3923,    374,    279,   6864,    315,   9822,     30,  12366,
            374,    539,    279,   4495,   4320,     11,    439,  12366,    374,
            264,   3682,   3363,    304,   9822,    719,    433,    374,    539,
            279]], device='cuda:0')


In [5]:
print(model.tokenizer.decode(result))

['<|begin_of_text|>What is the capital of France? Paris is not the correct answer, as Paris is a major city in France but it is not the']


### GPTQ Quantization

In [14]:
model_id = "PicoKittens/PicoMistral-23M"
quant_path = "PicoKittens/PicoMistral-23M-gptqmodel-4bit"

In [ ]:
from datasets import load_dataset
calibationDataset = load_dataset(
    "allenai/c4",
    data_files="en/c4-train.00000-of-01024.json.gz",
    split = "train"
).select(range(1024))["text"]

In [ ]:
from gptqmodel import GPTQModel, QuantizeConfig
quantConfig = QuantizeConfig(bits=4, group_size = 128)

In [ ]:
model = GPTQModel.load(model_id, quantConfig)

In [16]:
model.quantize(calibationDataset, batch_size=16)

INFO  Packing Kernel: selected: `TorchQuantLinear`                             


INFO  Packing Kernel: selected: `TorchQuantLinear`                             


INFO  Quantize: trimmed 307 calibration rows above max_position_embeddings=512 (longest original length=8711)


INFO  Calibration: Sort in descending order by length                          


INFO  Calibration: Total padded tokens: 3718                                   


...

INFO  +--------------------+-------+--------+-------+---------+--------+--------------------------------------------------+


INFO  | Process finalize   | 2     | 0.053  | 0.029 | 0.058   | 0.0%   | tp-pre-pad                                       |


INFO  +--------------------+-------+--------+-------+---------+--------+--------------------------------------------------+


> Quantizing self_attn.v_proj in layer ['p' to ||] [0 of 21] | 0:04:51 / 1:46:42
> Quantizing self_attn.k_proj in layer ['p' to ||] [0 of 23] | 0:02:41 / 1:04:24

{'gptq': [{'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.k_proj',
   'feat: in, out': '384, 128',
   'dtype: size': 'bf16: 0.1MB',
   'loss': '0.0000006291',
   'samples': '313002',
   'damp': '0.05000',
   'time': '0.698',
   'fwd_time': '1.013',
   '(v)ram': 'cuda 11.38G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.v_proj',
   'feat: in, out': '384, 128',
   'dtype: size': 'bf16: 0.1MB',
   'loss': '0.0000002650',
   'samples': '313002',
   'damp': '0.05000',
   'time': '0.760',
   'fwd_time': '1.013',
   '(v)ram': 'cuda 11.38G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.q_proj',
   'feat: in, out': '384, 384',
   'dtype: size': 'bf16: 0.3MB',
   'loss': '0.0000017860',
   'samples': '313002',
   'damp': '0.05000',
   'time': '0.802',
   'fwd_time': '1.013',
   '(v)ram': 'cuda 11.38G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.o_proj',
   'feat: in, out': '384, 384',
   'dtype: size': 'bf16: 0.3MB',
   'loss': '0.

In [17]:
model.save(quant_path)

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "quant_method": "gptq",
  "checkpoint_format": "gptq",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:5.7.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": 0.05,
    "damp_auto_increment": 0.01,
    "static_groups": false,
    "true_sequential": true,
    "mse": 0.0,
    "gptaq": null,
    "act_group_aware": true,
    "failsafe": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": {
        "type": "mad",
        "group_size_threshold": 128,
        "k": 2.75
      }
    },
    "offload_to_disk": true,
    "offload_to_disk_path": "./gptqmodel_offload/tnyjihzb-otaxzywi/",
    "pack_impl": "cpu",
    "mock_quantization": false,
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "hessian": {
      "chunk_size": null,
      "chunk_bytes": null

Files in directory:
generation_config.json
config.json
quant_log.csv
quantize_config.json
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 1,
    "do_sample": true,
    "eos_token_id": 2,
    "output_attentions": false,
    "output_hidden_states": false,
    "pad_token_id": 0,
    "transformers_version": "5.0.0",
    "use_cache": true
}
Content of saved `config.json`:
{
    "architectures": [
        "MistralForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 1,
    "dtype": "bfloat16",
    "eos_token_id": 2,
    "head_dim": 64,
    "hidden_act": "silu",
    "hidden_size": 384,
    "initializer_range": 0.02,
    "intermediate_size": 1536,
    "max_position_embeddings": 512,
    "model_type": "mistral",
    "num_attention_heads": 6,
    "num_hidden_layers": 8,
    "num_key_value_heads": 2,
    "pad_token_id": 0,
    "quantization_config": {
        "bits": 4,
        "checkpoint_format": "gptq",
        "desc_act": false,

INFO  Module: Sync lm_head <- from turtle (Linear)                             


INFO  Module: Re-tied embedding weights on shell model after full sync         


...

INFO  | Capture inputs     | 1     | 0.463  | 0.463 | 0.463   | 0.4%   | cache_inputs:MistralDecoderLayer                    |


INFO  +--------------------+-------+--------+-------+---------+--------+-----------------------------------------------------+


INFO  | Process finalize   | 2     | 0.053  | 0.029 | 0.058   | 0.0%   | tp-pre-pad                                          |


INFO  +--------------------+-------+--------+-------+---------+--------+-----------------------------------------------------+


> Quantizing self_attn.v_proj in layer ['p' to ||] [0 of 21] | 0:05:05 / 1:51:50
> Quantizing self_attn.k_proj in layer ['p' to ||] [0 of 23] | 0:02:55 / 1:10:00

In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import whoami
whoami()

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(
    repo_id="amxn18/PicoMistral-23M-gptqmodel-4bit",
    repo_type="model",
    private=False
)

upload_folder(
    repo_id="amxn18/PicoMistral-23M-gptqmodel-4bit",
    folder_path="./PicoKittens/PicoMistral-23M-gptqmodel-4bit",
    commit_message="Upload GPTQ quantized"
)

In [ ]:
import torch
import math
from datasets import load_dataset
from gptqmodel import GPTQModel

model_id = "PicoKittens/PicoMistral-23M"
quant_path = "PicoKittens/PicoMistral-23M-gptqmodel-4bit"

device = "cuda" if torch.cuda.is_available() else "cpu"


print("Loading evaluation dataset...")
eval_texts = load_dataset(
    "allenai/c4",
    data_files="en/c4-train.00000-of-01024.json.gz",
    split="train"
).select(range(200))["text"]


def calculate_avg_ppl(model_wrapper, texts, max_length=512):
    model = model_wrapper.model
    tokenizer = model_wrapper.tokenizer

    model.eval()
    model.tie_weights()

    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for text in texts:
            enc = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
            )

            input_ids = enc["input_ids"].to(device)

            outputs = model(
                input_ids=input_ids,
                labels=input_ids
            )

            loss = outputs.loss
            num_tokens = input_ids.numel()

            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)
    return ppl

print("Loading normal model...")
normal_model = GPTQModel.load(model_id, quantize_config=None)

print("Loading quantized model...")
quant_model = GPTQModel.load(quant_path, device=device)

In [31]:
print("Calculating PPL for normal model...")
ppl_normal = calculate_avg_ppl(normal_model, eval_texts)

print("Calculating PPL for quantized model...")
ppl_quant = calculate_avg_ppl(quant_model, eval_texts)


print("Normal Model PPL   :", ppl_normal)
print("Quantized Model PPL:", ppl_quant)

Calculating PPL for normal model...
Calculating PPL for quantized model...
Normal Model PPL   : 120.84557829020528
Quantized Model PPL: 121.98906718680107
